In [1]:
!pip install wandb

In [2]:
# from kaggle_secrets import UserSecretsClient
import wandb 

# user_secrets = UserSecretsClient()
# secret_value_0 = user_secrets.get_secret("wandb_key")

# Authenticate with W&B
wandb.login(key="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")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: quddusikashaf (quddusikashaf-habib-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import os
import glob
import random
import numpy as np
import scipy.io as scio
from scipy.signal import butter, sosfiltfilt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
from tqdm import tqdm


# Config

In [4]:
class Config:
    project_name = 'ninapro_db2_baselines'
    save_dir = '/kaggle/working/outputs'
    processed_dir = '/kaggle/working/db2_processed'
    base_path = '/kaggle/input/datasets/quddusikashaf/ninapro-db2'
    
    max_seq_len = 200 
    stride = 50
    num_channels = 12 
    num_angles = 10 
    
    batch_size = 64
    lr = 1e-4
    epochs = 30
    seed = 42
    
    # Model Hyperparameters
    d_model = 128
    d_ff = 256
    dropout = 0.2
    num_layers = 2 # For LSTM
    patience = 5

cfg = Config()
os.makedirs(cfg.save_dir, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reproducibility
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

# Data Processing

In [5]:
def mu_law_normalize(x, mu=255):
    return np.sign(x) * np.log1p(mu * np.abs(x)) / np.log1p(mu)

def bandpass_butterworth(emg, lowcut=5, highcut=500, fs=2000, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    sos = butter(order, [low, high], btype='band', output='sos')
    return sosfiltfilt(sos, emg, axis=0)

def process_dataset_for_oneSubject(subject, cfg):
    print(f"Processing subject: {subject}...")
    data_paths = sorted(glob.glob(f'{cfg.base_path}/{subject}/{subject}/*.mat'))
    if not data_paths:
        print(f"No MAT files found for {subject}")
        return
        
    emg_all, glove_all, restim_all = [], [], []
    for path in data_paths[:2]:  # use first two sessions
        data = scio.loadmat(path)
        emg_all.append(data['emg'])
        glove_all.append(data['glove'])
        restim_all.append(data['restimulus'])
    
    emg = np.vstack(emg_all).astype(np.float32)
    glove = np.vstack(glove_all).astype(np.float32)
    restimulus = np.vstack(restim_all).astype(np.int32).squeeze()

    emg = bandpass_butterworth(emg)
    emg = abs(emg)
    emg = emg / (np.max(np.abs(emg), axis=0, keepdims=True) + 1e-8)
    emg = mu_law_normalize(emg)

    glove_min = np.min(glove, axis=0, keepdims=True)
    glove_max = np.max(glove, axis=0, keepdims=True)
    glove = (glove - glove_min) / (glove_max - glove_min + 1e-8)

    cols = [1, 2, 4, 5, 7, 8, 11, 12, 15, 16]
    glove = glove[:, cols]
    
    emg_windows, glove_windows, glove_diff_windows, restim_windows = [], [], [], []
    for i in range(0, len(emg) - cfg.max_seq_len, cfg.stride):
        emg_win = emg[i:i+cfg.max_seq_len, :]
        glove_last = glove[i+cfg.max_seq_len-1, :]
        glove_first = glove[i, :]
        glove_diff_win = glove_last - glove_first 
        restim_win = restimulus[i+cfg.max_seq_len-1]
        
        emg_windows.append(emg_win)
        glove_windows.append(glove_last)
        glove_diff_windows.append(glove_diff_win)
        restim_windows.append(restim_win)

    output_dir = f"{cfg.processed_dir}/{subject}"
    os.makedirs(output_dir, exist_ok=True)
    np.savez(f"{output_dir}/data.npz", 
             sEMG=np.array(emg_windows), 
             angles=np.array(glove_windows), 
             angle_diff=np.array(glove_diff_windows),
             restim=np.array(restim_windows))
    print(f"Saved {len(emg_windows)} samples for {subject}")

In [6]:
# Run preprocessing for 3 subjects
subjects = sorted([f for f in os.listdir(cfg.base_path) if f.startswith('DB2')])

npz_files = []
# Loop over the first 3 subjects (index 0, 1, and 2)
for subject in subjects[:3]:
    process_dataset_for_oneSubject(subject, cfg)
    npz_files.append(f"{cfg.processed_dir}/{subject}/data.npz")

Processing subject: DB2_s1...
Saved 87229 samples for DB2_s1
Processing subject: DB2_s10...
Saved 87073 samples for DB2_s10
Processing subject: DB2_s11...
Saved 86991 samples for DB2_s11


# 3. DATASET & DATALOADER

In [7]:
class NinaproDataset(Dataset):
    def __init__(self, npz_files, split='train', split_ratio=0.8):
        sEMG_all, angles_all, angle_diff_all = [], [], []

        for f in npz_files:
            data = np.load(f, mmap_mode='r')
            sEMG, angles, angle_diff, restim = data['sEMG'], data['angles'], data['angle_diff'], data['restim']

            n_movement_1 = np.sum(restim == 1)
            for rep in np.unique(restim):
                idx = np.where(restim == rep)[0]
                if rep == 0: 
                    if len(idx) > n_movement_1:
                        np.random.shuffle(idx)
                        idx = idx[:n_movement_1] 
                idx.sort()

                split_idx = int(split_ratio * len(idx))
                selected_idx = idx[:split_idx] if split == 'train' else idx[split_idx:]
                if len(selected_idx) > 0:
                    sEMG_all.append(sEMG[selected_idx])
                    angles_all.append(angles[selected_idx])
                    angle_diff_all.append(angle_diff[selected_idx]) 

        self.sEMG = np.concatenate(sEMG_all, axis=0)
        self.angles = np.concatenate(angles_all, axis=0)
        self.angle_diff = np.concatenate(angle_diff_all, axis=0)

    def __len__(self): return len(self.sEMG)

    def __getitem__(self, idx):
        return (torch.tensor(self.sEMG[idx].T, dtype=torch.float32), # [Channels, Time]
                torch.tensor(self.angles[idx], dtype=torch.float32),
                torch.tensor(self.angle_diff[idx], dtype=torch.float32))

In [8]:
train_loader = DataLoader(NinaproDataset(npz_files, 'train'), batch_size=cfg.batch_size, shuffle=True)
test_loader = DataLoader(NinaproDataset(npz_files, 'test'), batch_size=cfg.batch_size, shuffle=False)

In [9]:
print(f"Train Dataset: {len(train_loader.dataset)}")
print(f"Test Dataset: {len(test_loader.dataset)}")


Train Dataset: 119926
Test Dataset: 30044


# 4. BASE MODELS

In [10]:
class BaselineMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(cfg.num_channels * cfg.max_seq_len, cfg.d_ff),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_ff, cfg.d_model),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model, cfg.num_angles)
        )
    def forward(self, x): return self.net(x)

class Baseline1DCNN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(cfg.num_channels, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, cfg.d_model, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), 
            nn.Flatten(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model, cfg.num_angles)
        )
    def forward(self, x): return self.net(x)

class BaselineLSTM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.lstm = nn.LSTM(input_size=cfg.num_channels, hidden_size=cfg.d_model, 
                            num_layers=cfg.num_layers, batch_first=True, dropout=cfg.dropout)
        self.fc = nn.Linear(cfg.d_model, cfg.num_angles)
    def forward(self, x):
        out, _ = self.lstm(x.transpose(1, 2))
        return self.fc(out[:, -1, :])

class BaselineConvLSTM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(cfg.num_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.lstm = nn.LSTM(input_size=64, hidden_size=cfg.d_model, num_layers=1, batch_first=True)
        self.fc = nn.Linear(cfg.d_model, cfg.num_angles)
    def forward(self, x):
        x = self.conv(x).transpose(1, 2)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

def get_model(model_name, cfg):
    if model_name == 'MLP': return BaselineMLP(cfg)
    elif model_name == 'CNN': return Baseline1DCNN(cfg)
    elif model_name == 'LSTM': return BaselineLSTM(cfg)
    elif model_name == 'ConvLSTM': return BaselineConvLSTM(cfg)

# 5. EVALUATION & PLOTTING


In [11]:
def evaluate_and_plot_joints(all_targets, all_preds, model_name, target_type, save_dir):
    joint_names = ['T_CMC_ABD', 'T_MCP_FLEX', 'WRIST_ABD', 'I_MCP_FLEX', 
                   'I_MCP_ABD', 'M_MCP_FLEX', 'R_MCP_FLEX', 'R_PIP_FLEX', 
                   'P_MCP_FLEX', 'P_PIP_FLEX']
    
    r2_scores, mse_scores = [], []
    for i in range(10):
        r2_scores.append(r2_score(all_targets[:, i], all_preds[:, i]))
        mse_scores.append(mean_squared_error(all_targets[:, i], all_preds[:, i]))
        
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    x = np.arange(10)
    
    ax1.bar(x, r2_scores, 0.5, color='#4C72B0', edgecolor='black')
    ax1.set_ylabel('R2 Score')
    ax1.set_title(f'{model_name} ({target_type.upper()}) - R2 by Joint')
    ax1.set_xticks(x); ax1.set_xticklabels(joint_names, rotation=30, ha='right')
    ax1.axhline(0, color='black'); ax1.set_ylim(-0.2, 1.0)
    
    ax2.bar(x, mse_scores, 0.5, color='#C44E52', edgecolor='black')
    ax2.set_ylabel('MSE')
    ax2.set_title(f'{model_name} ({target_type.upper()}) - MSE by Joint')
    ax2.set_xticks(x); ax2.set_xticklabels(joint_names, rotation=30, ha='right')
    
    plt.tight_layout()
    plot_path = os.path.join(save_dir, f"{model_name}_{target_type}_metrics.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()
    
    # Log plot to WandB
    wandb.log({f"{model_name}_{target_type}_Joint_Plot": wandb.Image(plot_path)})


# 6. TRAINING ENGINE


In [12]:
def run_experiment(model_name, target_type, cfg):
    print(f"\n{'='*40}\nStarting: {model_name} | Target: {target_type.upper()}\n{'='*40}")
    
    wandb.init(project=cfg.project_name, name=f"{model_name}_{target_type}", config=cfg.__dict__, reinit="finish_previous")
    
    model = get_model(model_name, cfg).to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    criterion = nn.MSELoss()
    
    scale_factor = 100.0 if target_type == 'diff' else 1.0
    best_rmse = float('inf')
    patience_cnt = 0
    
    for epoch in range(cfg.epochs):
        # --- TRAINING ---
        model.train()
        train_loss = 0
        for sEMG, angles, diffs in tqdm(train_loader, leave=False, desc=f"Epoch {epoch+1}"):
            sEMG = sEMG.to(device)
            target = diffs.to(device) if target_type == 'diff' else angles.to(device)
            target = target * scale_factor
            
            optimizer.zero_grad()
            outputs = model(sEMG)
            loss = criterion(outputs, target)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        # --- VALIDATION ---
        model.eval()
        val_loss = 0
        all_preds, all_targets = [], []
        with torch.no_grad():
            for sEMG, angles, diffs in test_loader:
                sEMG = sEMG.to(device)
                
                if target_type == 'diff':
                    target = diffs.to(device) * scale_factor
                    outputs = model(sEMG)
                    val_loss += criterion(outputs, target).item()
                    
                    # Reconstruct absolute position
                    pred_diff_real = outputs.cpu().numpy() / scale_factor
                    true_diff_real = diffs.numpy()
                    true_end_angles = angles.numpy()
                    
                    # True End - True Diff = True Start
                    true_start_angles = true_end_angles - true_diff_real 
                    
                    # Add predicted diff to true start
                    final_preds = true_start_angles + pred_diff_real
                    final_targets = true_end_angles
                    
                else:
                    target = angles.to(device)
                    outputs = model(sEMG)
                    val_loss += criterion(outputs, target).item()
                    
                    final_preds = outputs.cpu().numpy()
                    final_targets = angles.numpy()
                
                all_preds.append(final_preds)
                all_targets.append(final_targets)
                
        val_loss /= len(test_loader)
        all_preds = np.concatenate(all_preds)
        all_targets = np.concatenate(all_targets)
        
        val_rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
        val_r2 = r2_score(all_targets, all_preds)
        
        scheduler.step(val_rmse)
        wandb.log({"train_loss": train_loss, "val_loss": val_loss, "val_rmse": val_rmse, "val_r2": val_r2, "epoch": epoch+1})
        
        print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | RMSE: {val_rmse:.4f} | R2: {val_r2:.4f}")
        
        if val_rmse < best_rmse:
            best_rmse = val_rmse
            torch.save(model.state_dict(), f"{cfg.save_dir}/{model_name}_{target_type}_best.pt")
            patience_cnt = 0
            best_preds = all_preds
            best_targets = all_targets
        else:
            patience_cnt += 1
            if patience_cnt >= cfg.patience:
                print("Early stopping triggered.")
                break
                
    # Plot final best metrics
    evaluate_and_plot_joints(best_targets, best_preds, model_name, target_type, cfg.save_dir)
    wandb.finish()


# 7. EXECUTION GRID

In [13]:
models_to_test = ['MLP', 'CNN', 'LSTM', 'ConvLSTM']
targets_to_test = ['absolute', 'diff']

for target in targets_to_test:
    for model_id in models_to_test:
        run_experiment(model_id, target, cfg)

print("\nAll baseline experiments completed successfully!")


Starting: MLP | Target: ABSOLUTE


Epoch 01 | Train Loss: 0.0365 | Val Loss: 0.0230 | RMSE: 0.1515 | R2: 0.3712


Epoch 02 | Train Loss: 0.0239 | Val Loss: 0.0192 | RMSE: 0.1387 | R2: 0.4687


Epoch 03 | Train Loss: 0.0210 | Val Loss: 0.0175 | RMSE: 0.1322 | R2: 0.5142


Epoch 04 | Train Loss: 0.0196 | Val Loss: 0.0167 | RMSE: 0.1293 | R2: 0.5332


...

Epoch 28 | Train Loss: 5.5278 | Val Loss: 5.3895 | RMSE: 0.0232 | R2: 0.9843


Epoch 29 | Train Loss: 5.5207 | Val Loss: 5.3953 | RMSE: 0.0232 | R2: 0.9843


Epoch 30 | Train Loss: 5.5160 | Val Loss: 5.4501 | RMSE: 0.0233 | R2: 0.9841


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_loss,█▆▆▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁
val_loss,█▆▆▆▅▅▄▅▄▃▄▃▄▃▄▃▂▃▃▂▄▁▂▂▁▁▁▁▁▂
val_r2,▁▃▃▃▄▄▅▄▅▆▅▆▅▆▅▆▇▆▆▆▅██▇█████▇
val_rmse,█▆▆▆▅▅▄▅▄▃▄▃▄▃▄▃▂▃▃▃▄▁▂▂▁▁▁▁▁▂
epoch,30
train_loss,5.51605
val_loss,5.45013
val_r2,0.98414
val_rmse,0.02335



All baseline experiments completed successfully!
